#Analisis kondisi data: missing values, duplikat, outlier, imbalance, noise

In [1]:
import pandas as pd
import numpy as np

# Membaca dataset
datasets = {
    'pesticides.csv': pd.read_csv('pesticides.csv'),
    'rainfall.csv': pd.read_csv('rainfall.csv'),
    'temp.csv': pd.read_csv('temp.csv'),
    'yield.csv': pd.read_csv('yield.csv'),
    'yield_df.csv': pd.read_csv('yield_df.csv'),
    'crop_yield.csv': pd.read_csv('crop_yield.csv'),
    'crop_yield_dataset.csv': pd.read_csv('crop_yield_dataset.csv')
}

# Menangani format khusus untuk pemeriksaan
rainfall_col = 'average_rain_fall_mm_per_year'

datasets['rainfall.csv'][rainfall_col] = pd.to_numeric(
    datasets['rainfall.csv'][rainfall_col],
    errors='coerce'
)

# Pengaturan fitur untuk pemeriksaan outlier
outlier_features = {
    'pesticides.csv': ['Value'],
    'rainfall.csv': [rainfall_col],
    'temp.csv': ['avg_temp'],
    'yield.csv': ['Value'],
    'yield_df.csv': [
        'hg/ha_yield',
        'pesticides_tonnes',
        'avg_temp'
    ],
    'crop_yield.csv': [
        'Yield_tons_per_hectare'
    ],
    'crop_yield_dataset.csv': [
        'Yield_ton_per_ha'
    ]
}

# Target untuk pemeriksaan nilai negatif
targets = {
    'yield.csv': 'Value',
    'yield_df.csv': 'hg/ha_yield',
    'crop_yield.csv': 'Yield_tons_per_hectare',
    'crop_yield_dataset.csv': 'Yield_ton_per_ha'
}

# Pemeriksaan kondisi data
print("RINGKASAN KONDISI DATA")
print("-" * 75)

for nama, df in datasets.items():

    # Missing values
    missing = df.isnull().sum().sum()

    # Duplicate
    if nama == 'yield_df.csv':
        duplicate = df.drop(
            columns=['Unnamed: 0']
        ).duplicated().sum()
    else:
        duplicate = df.duplicated().sum()

    # Outliers menggunakan IQR
    total_outlier = 0

    for col in outlier_features[nama]:

        data = df[col].dropna()

        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1

        batas_bawah = q1 - 1.5 * iqr
        batas_atas = q3 + 1.5 * iqr

        total_outlier += (
            (data < batas_bawah) |
            (data > batas_atas)
        ).sum()

    # Nilai target negatif
    negatif = 0

    if nama in targets:
        negatif = (df[targets[nama]] < 0).sum()

    print(
        f"{nama:25} | "
        f"Missing: {missing:5} | "
        f"Duplikat: {duplicate:5} | "
        f"Outlier: {total_outlier:5} | "
        f"Negatif: {negatif:4}"
    )

# Pemeriksaan distribusi kategori
kategori = {
    'yield_df.csv': ['Area', 'Item'],
    'crop_yield.csv': ['Region', 'Soil_Type', 'Crop'],
    'crop_yield_dataset.csv': [
        'Crop', 'Region', 'Soil_Type'
    ]
}

print("\nDISTRIBUSI KATEGORI")
print("-" * 45)

for nama, kolom in kategori.items():

    for col in kolom:
        frekuensi = datasets[nama][col].value_counts()

        print(
            f"{nama} - {col}: "
            f"{frekuensi.min()} sampai {frekuensi.max()}"
        )

# Pemeriksaan indikator noise
print("\nINDIKASI NOISE")
print("-" * 45)

print(
    "Nilai '..' pada rainfall:",
    (pd.read_csv('rainfall.csv')[rainfall_col]
     .astype(str).str.strip() == '..').sum()
)

print("Pemeriksaan selesai.")

/tmp/ipykernel_1436/1473178431.py:11: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  'crop_yield.csv': pd.read_csv('crop_yield.csv'),


RINGKASAN KONDISI DATA
---------------------------------------------------------------------------
pesticides.csv            | Missing:     0 | Duplikat:     0 | Outlier:   615 | Negatif:    0
rainfall.csv              | Missing:   780 | Duplikat:     0 | Outlier:     0 | Negatif:    0
temp.csv                  | Missing:  2547 | Duplikat:  6958 | Outlier:     2 | Negatif:    0
yield.csv                 | Missing:     0 | Duplikat:     0 | Outlier:  3169 | Negatif:    0
yield_df.csv              | Missing:     0 | Duplikat:  2310 | Outlier:  3511 | Negatif:    0
crop_yield.csv            | Missing:     6 | Duplikat:     0 | Outlier:    16 | Negatif:   49
crop_yield_dataset.csv    | Missing:  4569 | Duplikat:     0 | Outlier:     0 | Negatif:    0

DISTRIBUSI KATEGORI
---------------------------------------------
yield_df.csv - Area: 24 sampai 4048
yield_df.csv - Item: 556 sampai 4276
crop_yield.csv - Region: 47562 sampai 47893
crop_yield.csv - Soil_Type: 31694 sampai 31941
crop_yield.c

#Data cleaning dan feature engineering

Data Cleaning


In [2]:
import pandas as pd
import numpy as np

# Membaca dataset
datasets = {
    'pesticides.csv': pd.read_csv('pesticides.csv'),
    'rainfall.csv': pd.read_csv('rainfall.csv'),
    'temp.csv': pd.read_csv('temp.csv'),
    'yield.csv': pd.read_csv('yield.csv'),
    'yield_df.csv': pd.read_csv('yield_df.csv'),
    'crop_yield.csv': pd.read_csv('crop_yield.csv'),
    'crop_yield_dataset.csv': pd.read_csv('crop_yield_dataset.csv')
}

cleaned = {}

for nama, data in datasets.items():

    df = data.copy()
    df.columns = df.columns.str.strip()

    # Menghapus kolom indeks tambahan
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    # Konversi data numerik
    if nama == 'rainfall.csv':
        col = 'average_rain_fall_mm_per_year'
        df[col] = pd.to_numeric(df[col], errors='coerce')

        # Imputasi median berdasarkan wilayah
        df[col] = df[col].fillna(
            df.groupby('Area')[col].transform('median')
        )

        # Jika masih kosong, gunakan median keseluruhan
        df[col] = df[col].fillna(df[col].median())

    elif nama == 'temp.csv':
        df['avg_temp'] = pd.to_numeric(
            df['avg_temp'], errors='coerce'
        )

        # Menghapus data duplikat
        df = df.drop_duplicates()

        # Imputasi median berdasarkan negara
        df['avg_temp'] = df['avg_temp'].fillna(
            df.groupby('country')['avg_temp'].transform('median')
        )

        df['avg_temp'] = df['avg_temp'].fillna(
            df['avg_temp'].median()
        )

    elif nama == 'pesticides.csv':
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

    elif nama == 'yield.csv':
        for col in ['Year', 'Value']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

    elif nama == 'yield_df.csv':
        numeric_cols = [
            'Year',
            'hg/ha_yield',
            'average_rain_fall_mm_per_year',
            'pesticides_tonnes',
            'avg_temp'
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Menghapus baris duplikat
        df = df.drop_duplicates()

    elif nama == 'crop_yield.csv':
        numeric_cols = [
            'Rainfall_mm',
            'Temperature_Celsius',
            'Days_to_Harvest',
            'Yield_tons_per_hectare'
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Menghapus nilai target negatif
        df = df[df['Yield_tons_per_hectare'] >= 0]

        df = df.drop_duplicates()

    elif nama == 'crop_yield_dataset.csv':
        numeric_cols = [
            'Soil_pH',
            'Rainfall_mm',
            'Temperature_C',
            'Humidity_pct',
            'Fertilizer_Used_kg',
            'Pesticides_Used_kg',
            'Planting_Density',
            'Yield_ton_per_ha'
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Mengisi missing values kategorikal
        for col in ['Irrigation', 'Previous_Crop']:
            df[col] = df[col].fillna('Unknown')

        df = df.drop_duplicates()

    cleaned[nama] = df.copy()


# Ringkasan hasil cleaning
print("HASIL DATA CLEANING")
print("-" * 65)

for nama, df in cleaned.items():
    print(
        f"{nama:25} | "
        f"Baris: {len(df):7} | "
        f"Missing: {df.isnull().sum().sum():5}"
    )

print("\nData cleaning selesai.")

/tmp/ipykernel_1436/15195691.py:11: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  'crop_yield.csv': pd.read_csv('crop_yield.csv'),


HASIL DATA CLEANING
-----------------------------------------------------------------
pesticides.csv            | Baris:    4349 | Missing:     0
rainfall.csv              | Baris:    6727 | Missing:     0
temp.csv                  | Baris:   64353 | Missing:     0
yield.csv                 | Baris:   56717 | Missing:     0
yield_df.csv              | Baris:   25932 | Missing:     0
crop_yield.csv            | Baris:  235693 | Missing:     0
crop_yield_dataset.csv    | Baris:   10000 | Missing:     0

Data cleaning selesai.


feature engineering

In [3]:
# Mengambil dataset utama yang sudah dibersihkan
feature_data = {
    'Dataset 1': cleaned['yield_df.csv'].copy(),
    'Dataset 2': cleaned['crop_yield.csv'].copy(),
    'Dataset 3': cleaned['crop_yield_dataset.csv'].copy()
}

# 1. Creating New Features

# Dataset 1
feature_data['Dataset 1']['Rainfall_Temp_Interaction'] = (
    feature_data['Dataset 1']['average_rain_fall_mm_per_year'] *
    feature_data['Dataset 1']['avg_temp']
)

# Dataset 2
feature_data['Dataset 2']['Rainfall_Temp_Interaction'] = (
    feature_data['Dataset 2']['Rainfall_mm'] *
    feature_data['Dataset 2']['Temperature_Celsius']
)

# Dataset 3
feature_data['Dataset 3']['Rainfall_Temp_Interaction'] = (
    feature_data['Dataset 3']['Rainfall_mm'] *
    feature_data['Dataset 3']['Temperature_C']
)


# 2. Feature Selection

features = {}
targets = {}

feature_columns = {
    'Dataset 1': [
        'Area', 'Item', 'Year',
        'average_rain_fall_mm_per_year',
        'pesticides_tonnes',
        'avg_temp',
        'Rainfall_Temp_Interaction'
    ],

    'Dataset 2': [
        'Region', 'Soil_Type', 'Crop',
        'Rainfall_mm', 'Temperature_Celsius',
        'Fertilizer_Used', 'Irrigation_Used',
        'Weather_Condition', 'Days_to_Harvest',
        'Rainfall_Temp_Interaction'
    ],

    'Dataset 3': [
        'Crop', 'Region', 'Soil_Type', 'Soil_pH',
        'Rainfall_mm', 'Temperature_C',
        'Humidity_pct', 'Fertilizer_Used_kg',
        'Irrigation', 'Pesticides_Used_kg',
        'Planting_Density', 'Previous_Crop',
        'Rainfall_Temp_Interaction'
    ]
}

target_columns = {
    'Dataset 1': 'hg/ha_yield',
    'Dataset 2': 'Yield_tons_per_hectare',
    'Dataset 3': 'Yield_ton_per_ha'
}

for nama, df in feature_data.items():

    features[nama] = df[feature_columns[nama]].copy()
    targets[nama] = df[target_columns[nama]].copy()


# 3. Ringkasan hasil Feature Engineering

print("HASIL FEATURE ENGINEERING")
print("-" * 60)

for nama in feature_data:

    print(f"{nama}")
    print("Jumlah data:", len(feature_data[nama]))
    print("Jumlah fitur:", features[nama].shape[1])
    print("Target:", target_columns[nama])
    print("Fitur baru: Rainfall_Temp_Interaction")
    print("-" * 60)


# 4. Menyimpan hasil data preparation

for nama, df in feature_data.items():

    filename = nama.lower().replace(" ", "_")

    df.to_csv(
        f'{filename}_prepared.csv',
        index=False
    )

print("Data preparation berhasil disimpan.")

HASIL FEATURE ENGINEERING
------------------------------------------------------------
Dataset 1
Jumlah data: 25932
Jumlah fitur: 7
Target: hg/ha_yield
Fitur baru: Rainfall_Temp_Interaction
------------------------------------------------------------
Dataset 2
Jumlah data: 235693
Jumlah fitur: 10
Target: Yield_tons_per_hectare
Fitur baru: Rainfall_Temp_Interaction
------------------------------------------------------------
Dataset 3
Jumlah data: 10000
Jumlah fitur: 13
Target: Yield_ton_per_ha
Fitur baru: Rainfall_Temp_Interaction
------------------------------------------------------------
Data preparation berhasil disimpan.
